In [9]:
import duckdb

con = duckdb.connect("../warehouse/olist.duckdb", read_only=True)
q = lambda sql: con.sql(sql).df()   # run SQL, get a pandas DataFrame

q("SELECT count(*) AS orders FROM raw.orders")

,orders
0,99441


In [10]:
q("SUMMARIZE raw.orders")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,99974,NaN,None,NaN,NaN,NaN,99441,0.00
1,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,97769,NaN,None,NaN,NaN,NaN,99441,0.00
2,order_status,VARCHAR,approved,unavailable,8,NaN,None,NaN,NaN,NaN,99441,0.00
3,order_purchase_timestamp,TIMESTAMP,2016-09-04 21:15:19,2018-10-17 17:30:18,96571,2017-12-31 08:43:12.776581,None,2017-09-12 09:46:15.721942,2018-01-19 08:25:16.131472,2018-05-04 21:22:52.465787,99441,0.00
4,order_approved_at,TIMESTAMP,2016-09-15 12:16:38,2018-09-03 17:40:06,87279,2017-12-31 18:35:24.0988,None,2017-09-12 23:38:47.244113,2018-01-19 19:31:11.91185,2018-05-05 10:39:06.448894,99441,0.16
5,order_delivered_carrier_date,TIMESTAMP,2016-10-08 10:34:01,2018-09-11 19:48:28,81859,2018-01-04 21:49:48.138278,None,2017-09-17 05:01:14.896067,2018-01-24 13:03:07.097667,2018-05-08 03:14:38.982959,99441,1.79
6,order_delivered_customer_date,TIMESTAMP,2016-10-11 13:46:32,2018-10-17 13:22:46,93526,2018-01-14 12:09:19.035543,None,2017-09-25 19:42:45.079512,2018-02-03 08:46:48.115237,2018-05-16 04:30:00.070728,99441,2.98
7,order_estimated_delivery_date,TIMESTAMP,2016-09-30 00:00:00,2018-11-12 00:00:00,451,2018-01-24 03:08:37.730111,None,2017-10-03 07:03:45.764487,2018-02-14 12:08:43.90428,2018-05-26 07:48:29.412612,99441,0.00


In [11]:
q("""
SELECT date_trunc('month', order_purchase_timestamp) AS month,
       count(*) AS orders
FROM raw.orders
GROUP BY 1
ORDER BY 1
""")

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [13]:
q("""
SELECT order_status, count(*) AS orders
FROM raw.orders
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## Findings: order timeline and status

**Date range.** Orders run from September 2016 to October 2018, but the first and last months are incomplete. 2016 has only 329 orders in total (November 2016 has none), and September–October 2018 have just 20. All trend analysis will therefore use **January 2017 to August 2018**, which gives 20 complete months.

**Growth.** Monthly orders grew about 9x in a year, from 800 in January 2017 to 7,269 in January 2018. Growth then levelled off, with 2018 months between 6,167 and 7,269 orders. November 2017 is the peak at 7,544 orders, 63% above October, most likely due to Black Friday (24 November 2017).

**Order status.** 97.0% of orders (96,478 of 99,441) were delivered. 1.2% failed (625 canceled and 609 unavailable) and will be excluded from revenue. The remaining 1.7% (1,729 orders: shipped, invoiced, processing, created or approved) were still in progress when the data was extracted, so they count as sales but have no delivery date.

In [14]:
q("""
SELECT count(DISTINCT customer_id)        AS customer_ids,
       count(DISTINCT customer_unique_id) AS real_customers
FROM raw.customers
""")

,customer_ids,real_customers
0,99441,96096


In [15]:
q("""
WITH per_customer AS (
    SELECT c.customer_unique_id,
           count(DISTINCT o.order_id) AS n_orders
    FROM raw.orders o
    JOIN raw.customers c USING (customer_id)
    GROUP BY 1
)
SELECT count(*) AS customers,
       round(100.0 * avg(CASE WHEN n_orders = 1 THEN 1 ELSE 0 END), 1) AS pct_one_time,
       max(n_orders) AS most_orders_by_one_customer
FROM per_customer
""")

,customers,pct_one_time,most_orders_by_one_customer
0,96096,96.9,17


## Gotcha 1: `customer_id` is not a customer

There are 99,441 distinct `customer_id` values, exactly one per order, because Olist creates a new ID for every order. The real person is identified by `customer_unique_id`, which has 96,096 distinct values.

**96.9% of customers ordered only once.** Only about 3.1% (roughly 2,980 customers) made a repeat purchase, and the most frequent customer placed 17 orders.

**Decisions:**
- All customer-level analysis uses `customer_unique_id`. Using `customer_id` would wrongly show a 0% repeat rate.
- With so few repeat buyers, a churn model has almost nothing to learn from. The predictive model will instead flag orders likely to receive a 1–2 star review.

In [16]:
q("""
SELECT count(*)                  AS review_rows,
       count(DISTINCT review_id) AS unique_review_ids,
       count(DISTINCT order_id)  AS orders_reviewed
FROM raw.order_reviews
""")

,review_rows,unique_review_ids,orders_reviewed
0,99224,98410,98673


In [17]:
q("""
WITH items AS (
    SELECT order_id, sum(price + freight_value) AS items_total
    FROM raw.order_items GROUP BY 1
),
pays AS (
    SELECT order_id, sum(payment_value) AS paid_total, count(*) AS payment_rows
    FROM raw.order_payments GROUP BY 1
)
SELECT count(*)                                                    AS orders,
       sum(CASE WHEN abs(items_total - paid_total) > 1 THEN 1 ELSE 0 END) AS mismatched,
       sum(CASE WHEN payment_rows > 1 THEN 1 ELSE 0 END)           AS multi_payment_orders
FROM items JOIN pays USING (order_id)
""")

,orders,mismatched,multi_payment_orders
0,98665,249.0,2936.0


## Gotcha 2: reviews are not one per order

There are 99,224 review rows but only 98,673 reviewed orders, so 551 rows are additional reviews for orders that already have one. There are also only 98,410 distinct `review_id` values, meaning some reviews are attached to more than one order. 768 orders have no review.

**Decisions:**
- Keep only the **latest review per order** (deduplicated on `order_id` in the staging layer).
- Do not use `review_id` as a unique key.

## Gotcha 3: payments and items fan out

2,936 orders (3.0%) have more than one payment row, and many orders have more than one item. Joining both tables to orders at the same time would multiply rows and inflate revenue. Item totals (price + freight) and payment totals differ by more than R$1 for 249 orders (0.25%), mostly due to instalment interest.

**Decisions:**
- Aggregate items and payments to **one row per order before joining**.
- Revenue is defined as **item price + freight** from the items table, for orders that were not canceled or unavailable.
- A dbt test will check that total revenue in the fact table equals the item-level total, so any future fan-out is caught.

In [18]:
q("""
SELECT order_status,
       count(*) AS orders,
       count(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS missing_delivery_date
FROM raw.orders
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders,missing_delivery_date
0,delivered,96478,8
1,shipped,1107,1107
2,canceled,625,619
3,unavailable,609,609
4,invoiced,314,314
5,processing,301,301
6,created,5,5
7,approved,2,2


In [19]:
q("""
SELECT p.product_category_name,
       count(*) AS products
FROM raw.products p
LEFT JOIN raw.category_translation t USING (product_category_name)
WHERE t.product_category_name_english IS NULL
GROUP BY 1
ORDER BY 2 DESC
""")

,product_category_name,products
0,NaN,610
1,portateis_cozinha_e_preparadores_de_alimentos,10
2,pc_gamer,3


In [20]:
q("""
SELECT count(*)                                   AS rows,
       count(DISTINCT geolocation_zip_code_prefix) AS zip_prefixes,
       max(n)                                     AS max_rows_for_one_prefix
FROM (
    SELECT geolocation_zip_code_prefix, count(*) OVER (PARTITION BY geolocation_zip_code_prefix) AS n
    FROM raw.geolocation
)
""")

,rows,zip_prefixes,max_rows_for_one_prefix
0,1000163,19015,1146


## Gotcha 4: missing and unexpected delivery dates

8 orders marked `delivered` have no delivery date, and 6 `canceled` orders do have one (probably canceled after delivery). All other non-delivered statuses have no delivery date, as expected.

**Decision:** exclude orders without a delivery date from delivery-time metrics only. They still count as sales.

## Gotcha 5: missing categories and duplicate locations

610 products have no category, and 2 categories have no English translation (`portateis_cozinha_e_preparadores_de_alimentos`, 10 products, and `pc_gamer`, 3 products). The geolocation table has 1,000,163 rows for only 19,015 zip prefixes, up to 1,146 rows for a single prefix.

**Decisions:**
- Category = English name, falling back to the Portuguese name, then `unknown`.
- Average latitude and longitude to one row per zip prefix before joining, and drop points outside Brazil.

## Data-quality log

| Issue | Evidence | Decision |
| --- | --- | --- |
| `customer_id` is per order | 99,441 `customer_id` vs 96,096 `customer_unique_id` | Use `customer_unique_id` for all customer analysis |
| Very low repeat rate | 96.9% of customers ordered once | Model bad reviews instead of churn |
| Duplicate reviews | 99,224 review rows for 98,673 orders; 98,410 review IDs | Keep the latest review per order |
| Payments and items fan out | 2,936 multi-payment orders; 249 totals differ by > R$1 | Aggregate to one row per order before joining; revenue = price + freight |
| Inconsistent delivery dates | 8 delivered orders without a date; 6 canceled orders with one | Exclude from delivery metrics only |
| Missing or untranslated categories | 610 products with no category; 2 untranslated categories (13 products) | Fall back to Portuguese name, then `unknown` |
| Many coordinates per zip prefix | 1,000,163 rows for 19,015 prefixes (max 1,146) | Average lat/lng per prefix; drop points outside Brazil |
| Sparse edge months | 329 orders in 2016; 20 in Sep–Oct 2018 | Trend analysis uses Jan 2017 to Aug 2018 |